In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
spend_raw = pd.read_excel('Spend (Done).xlsx')
spend = spend_raw.copy()

In [3]:
spend.head()

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-07-03,Facebook Ads,NaN,0,0.00,0,NaN,NaN
3,2023-07-03,Google Ads,NaN,0,0.00,0,NaN,NaN
4,2023-07-03,CRM,NaN,0,0.00,0,NaN,NaN


### # Verarbeitung von Marketingausgaben: Formatierung, Bereinigung und Datenextraktion

In [4]:
# 1. Formatierung von Datums- und Zahlenwerten
# Umwandlung der Spalte 'Date' in datetime und 'Spend' in numerische Werte (Fehler werden zu NaN und mit 0 aufgefüllt)
spend['Date'] = pd.to_datetime(spend['Date'])
spend['Spend'] = pd.to_numeric(spend['Spend'], errors='coerce').fillna(0)

# 2. Bereinigung der Quellen (Source)
# Umwandlung in Kleinbuchstaben, Trimming von Leerzeichen und Ersetzen von Fehlwerten durch 'unknown'
spend['Source'] = spend['Source'].str.lower().str.strip().fillna('unknown')

# 3. Verarbeitung von Kampagnennamen
# Extraktion des Startdatums (Format 00.00.00) in eine separate Spalte
spend['Campaign_Launch_Date'] = spend['Campaign'].str.extract(r'(\d{2}\.\d{2}\.\d{2})')
spend['Campaign_Launch_Date'] = pd.to_datetime(spend['Campaign_Launch_Date'], format='%d.%m.%y', errors='coerce')

# Erstellung eines bereinigten Campaign_Name durch Entfernen des Datums aus dem String
spend['Campaign_Name'] = spend['Campaign'].str.replace(r'\d{2}\.\d{2}\.\d{2}', '', regex=True).str.strip()

# Falls der String nach Entfernung des Datums leer ist, wird 'unknown' eingetragen
spend['Campaign_Name'] = spend['Campaign_Name'].replace('', 'unknown').fillna('unknown')

# 4. Entfernen nicht benötigter Spalten
# Löschen der alten Spalte 'Campaign' und der temporären Spalte 'Campaign_Name_Clean' (falls vorhanden)
cols_to_drop = ['Campaign', 'Campaign_Name_Clean']
spend.drop(columns=[c for c in cols_to_drop if c in spend.columns], inplace=True)

# 5. Optimierung der Datentypen
# Umwandlung von Textspalten in den Datentyp 'category' zur Speicheroptimierung
spend['Source'] = spend['Source'].astype('category')
spend['Campaign_Name'] = spend['Campaign_Name'].astype('category')

# # Überprüfung des Ergebnisses
print("--- Die ersten 10 Zeilen der aktualisierten Spend-Daten ---")
print(spend.head(10))
print("\n--- Finale Datenstruktur ---")
print(spend.info())

--- Die ersten 10 Zeilen der aktualisierten Spend-Daten ---
        Date        Source  Impressions  Spend  Clicks AdGroup   Ad  \
0 2023-07-03    google ads            6   0.00       0     NaN  NaN   
1 2023-07-03    google ads            4   0.01       1     NaN  NaN   
2 2023-07-03  facebook ads            0   0.00       0     NaN  NaN   
3 2023-07-03    google ads            0   0.00       0     NaN  NaN   
4 2023-07-03           crm            0   0.00       0     NaN  NaN   
5 2023-07-03  facebook ads          187   3.30       6   women   b3   
6 2023-07-03  facebook ads            4   0.02       1   women   b1   
7 2023-07-03      bloggers            0   0.00       0     NaN  NaN   
8 2023-07-03   youtube ads            0   0.00       0     NaN  NaN   
9 2023-07-03  facebook ads           61   0.58       0    wide   b4   

  Campaign_Launch_Date          Campaign_Name  
0                  NaT         gen_analyst_DE  
1                  NaT  performancemax_eng_DE  
2             

In [5]:
print(spend[['Campaign_Name', 'Campaign_Launch_Date', 'Spend']].head(10))

           Campaign_Name Campaign_Launch_Date  Spend
0         gen_analyst_DE                  NaT   0.00
1  performancemax_eng_DE                  NaT   0.01
2                unknown                  NaT   0.00
3                unknown                  NaT   0.00
4                unknown                  NaT   0.00
5                  women           2023-07-03   3.30
6                  women           2023-07-03   0.02
7                unknown                  NaT   0.00
8                unknown                  NaT   0.00
9                wide_DE           2023-07-02   0.58


In [5]:
spend.to_csv('Spend_Final.csv', index=False)